# Raster Geospatial Data

## Overview

Raster data represents continuous spatial fields as regular grids of cells. Each cell has a value (elevation, NDVI, temperature) and an associated geographic extent and resolution.

**Core raster concepts:**

| Concept | Meaning |
|---|---|
| Resolution | Cell size in CRS units (e.g. 25m for DTM) |
| Extent | Bounding box of the raster |
| NoData | Cells with missing values |
| Bands | Layers in a multi-band raster (e.g. RGB, multispectral) |
| Affine transform | Maps pixel row/column to real-world coordinates |

**Common operations:** clip to extent, resample to new resolution, zonal statistics (mean value per polygon), raster algebra, band compositing.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point, Polygon, mapping
import tempfile, os, warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# Simulate a 200x200 cell DTM (Digital Terrain Model) over a river catchment
# BNG EPSG:27700, resolution=25m, extent covers our monitoring sites
nrows, ncols = 200, 200
west, south, east, north = 317000, 731000, 327000, 736000   # BNG metres
transform = from_bounds(west, south, east, north, ncols, nrows)

# Create realistic elevation surface with river valley
x_grid = np.linspace(0, 1, ncols)
y_grid = np.linspace(0, 1, nrows)
XX, YY = np.meshgrid(x_grid, y_grid)
# Base: gently sloping upland
elev = 50 + 300 * (1 - YY) + 100 * XX
# River valley: lower elevation along central N-S corridor
valley = np.exp(-((XX - 0.5)**2) / 0.02)
elev -= 80 * valley
# Add terrain roughness
elev += rng.normal(0, 8, (nrows, ncols))
elev = elev.clip(20, 420).astype(np.float32)

print(f"DTM grid: {nrows}×{ncols} cells, resolution=25m")
print(f"Elevation range: {elev.min():.1f}–{elev.max():.1f} m")
print(f"Transform: {transform}")

---
## Writing and Reading Rasters

In [ ]:
# Write raster to GeoTIFF using rasterio
tmp_dtm = os.path.join(tempfile.gettempdir(), 'dtm_demo.tif')
with rasterio.open(
    tmp_dtm, 'w',
    driver='GTiff',
    height=nrows, width=ncols,
    count=1,           # 1 band
    dtype='float32',
    crs='EPSG:27700',
    transform=transform,
    nodata=-9999,
) as dst:
    dst.write(elev, 1)   # band index 1

# Read it back
with rasterio.open(tmp_dtm) as src:
    dtm_data = src.read(1)
    dtm_meta = src.meta
    dtm_transform = src.transform

print(f"Written and read back: {dtm_meta['width']}×{dtm_meta['height']}, dtype={dtm_meta['dtype']}")
print(f"CRS: {dtm_meta['crs']}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13,5))
im0 = axes[0].imshow(dtm_data, cmap='terrain', origin='upper')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)', shrink=0.8)
axes[0].set_title('Simulated DTM (25m resolution)')
axes[1].hist(dtm_data.ravel(), bins=60, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Elevation (m)'); axes[1].set_ylabel('Cell count')
axes[1].set_title('Elevation Distribution')
plt.tight_layout(); plt.show()

---
## Derived Rasters: Slope and NDVI

In [ ]:
# Derive slope from DTM using finite differences
def compute_slope(dtm, cell_size=25.0):
    dz_dy, dz_dx = np.gradient(dtm, cell_size, cell_size)
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    return np.degrees(slope_rad).astype(np.float32)

slope = compute_slope(dtm_data, cell_size=25.0)
print(f"Slope range: {slope.min():.1f}–{slope.max():.1f} degrees")

# Simulate NDVI (Normalised Difference Vegetation Index)
# NDVI = (NIR - Red) / (NIR + Red), range [-1, 1]
# Valleys tend to have higher NDVI (more riparian vegetation)
ndvi_base = 0.6 - 0.001*(dtm_data - 50)   # lower elevation -> higher NDVI
ndvi_base = np.clip(ndvi_base, 0.1, 0.9)
ndvi = ndvi_base + rng.normal(0, 0.05, ndvi_base.shape)
ndvi = np.clip(ndvi, -1, 1).astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, data, title, cmap, label in [
    (axes[0], dtm_data, 'DTM',    'terrain', 'Elevation (m)'),
    (axes[1], slope,    'Slope',  'hot_r',   'Slope (°)'),
    (axes[2], ndvi,     'NDVI',   'RdYlGn',  'NDVI'),
]:
    im = ax.imshow(data, cmap=cmap, origin='upper')
    plt.colorbar(im, ax=ax, label=label, shrink=0.8)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
print(f"NDVI range: {ndvi.min():.3f}–{ndvi.max():.3f}, mean={ndvi.mean():.3f}")

---
## Zonal Statistics

In [ ]:
# Zonal statistics: mean NDVI and elevation per catchment polygon
catchment_polygons = {
    'North': Polygon([(318000,733500),(323000,733500),(323000,736000),(318000,736000)]),
    'South': Polygon([(323000,733500),(327000,733500),(327000,736000),(323000,736000)]),
    'Main':  Polygon([(318000,731000),(327000,731000),(327000,733500),(318000,733500)]),
}

def zonal_stats(raster, raster_transform, polygon, stat='mean'):
    from rasterio.features import geometry_mask
    mask = geometry_mask([mapping(polygon)], out_shape=raster.shape,
                          transform=raster_transform, invert=True)
    values = raster[mask]
    values = values[~np.isnan(values)]
    if len(values) == 0: return np.nan
    return {'mean': np.mean, 'std': np.std, 'min': np.min, 'max': np.max}[stat](values)

print("Zonal statistics per catchment:")
print(f"{'Catchment':10s} {'Mean elev':>12} {'Mean NDVI':>11} {'Mean slope':>12}")
for name, poly in catchment_polygons.items():
    me = zonal_stats(dtm_data, dtm_transform, poly, 'mean')
    mn = zonal_stats(ndvi,     dtm_transform, poly, 'mean')
    ms = zonal_stats(slope,    dtm_transform, poly, 'mean')
    print(f"{name:10s} {me:12.1f} {mn:11.3f} {ms:12.2f}")

In [ ]:
# Clip raster to study area and resample
from rasterio.enums import Resampling

# Simulate 50m resampled version (2x coarser) by block averaging
def resample_raster(data, factor):
    h, w = data.shape
    nh, nw = h // factor, w // factor
    return data[:nh*factor, :nw*factor].reshape(nh, factor, nw, factor).mean(axis=(1,3))

dtm_50m = resample_raster(dtm_data, 2)
print(f"Original: {dtm_data.shape} at 25m")
print(f"Resampled: {dtm_50m.shape} at 50m")
print(f"Mean elevation preserved: {dtm_data.mean():.2f} vs {dtm_50m.mean():.2f}")

# Multi-band raster: stack DTM, slope, NDVI as 3-band raster
multiband = np.stack([
    (dtm_data - dtm_data.mean()) / dtm_data.std(),
    (slope - slope.mean()) / slope.std(),
    (ndvi - ndvi.mean()) / ndvi.std(),
], axis=0)
print(f"\nMulti-band raster shape: {multiband.shape}  (bands, rows, cols)")
print("Bands: [0]=standardised elevation, [1]=slope, [2]=NDVI")
fig, axes = plt.subplots(1,3, figsize=(13,4))
titles = ['Elevation (std)', 'Slope (std)', 'NDVI (std)']
for ax, band, title in zip(axes, multiband, titles):
    im = ax.imshow(band, cmap='RdBu_r', origin='upper')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Reading a raster and forgetting to handle NoData values**  
Rasterio stores NoData as a sentinel value (e.g. -9999, -32768). If not replaced with `np.nan` before analysis, NoData cells will be included in statistics, distorting means and ranges. After reading: `data = src.read(1).astype(float); data[data == src.nodata] = np.nan`.

**2. Computing zonal statistics without checking CRS alignment between raster and polygon**  
Zonal statistics use the raster's affine transform to map pixel positions to real-world coordinates. If the polygon is in a different CRS than the raster, zonal statistics will silently include the wrong cells or return empty results. Always ensure polygon and raster share the same CRS before masking.

**3. Resampling to a coarser resolution using nearest-neighbour for continuous data**  
Nearest-neighbour resampling assigns the value of the nearest cell without interpolation — appropriate for categorical data (land cover class) but produces a "blocky" appearance for continuous data (elevation, NDVI). Use bilinear or cubic resampling for continuous rasters with `resampling=Resampling.bilinear`.

**4. Using `np.gradient` for slope without accounting for cell size**  
`np.gradient(dtm)` returns finite differences in pixel units (dimensionless). Dividing by cell_size (e.g. 25m) gives the correct rise-over-run slope. Omitting the cell size produces slope values that are 25× too large for a 25m DTM. Always pass `edge_order` and the correct spacing to `np.gradient`.

**5. Loading entire large rasters into memory**  
A 1m-resolution DTM of a large catchment may be several GB. Loading it fully with `src.read()` exhausts memory. Use windowed reading (`src.read(1, window=...)`) or process in tiles. For analysis workflows, consider `rioxarray` for lazy-loaded, chunked raster processing.
---
*python_methods_library - Samantha McGarrigle*